In [1]:
pip install opencv-python


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install ultralytics

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install numpy==1.26.4

  Using cached numpy-1.26.4.tar.gz (15.8 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [21 lines of output]
      + c:\Users\user\AppData\Local\Programs\Python\Python313\python.exe C:\Users\user\AppData\Local\Temp\pip-install-o331oosx\numpy_26b454f355e846dba806e4db58bfdfc9\vendored-meson\meson\meson.py setup C:\Users\user\AppData\Local\Temp\pip-install-o331oosx\numpy_26b454f355e846dba806e4db58bfdfc9 C:\Users\user\AppData\Local\Temp\pip-install-o331oosx\numpy_26b454f355e846dba806e4db58bfdfc9\.mesonpy-gasfpi98 -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\user\AppData\Local\Temp\pip-install-o331oosx\numpy_26b454f355e846dba806e4db58bfdfc9\.mesonpy-gasfpi98\meson-python-native-file.ini
      The Meson build system
      Version: 1.2.99
      Source dir: C:\Users\user\AppData\Local\Temp\pip-install-o331oosx\numpy_26b454f355e846dba806e4db58bfdfc9
      Build dir: C:\Users\user\AppData\Local\Temp\pip-install-o331oosx\n

In [4]:
import numpy as np
print(np.__version__)

import cv2
from ultralytics import YOLO

print("Everything works!")

2.2.6
Everything works!


In [5]:
import cv2
import torch
from ultralytics import YOLO
from pathlib import Path

# Load model

BASE_DIR = Path.cwd()

YOLO_PATH = BASE_DIR / "YOLO26n_best.pt"
model = YOLO(str(YOLO_PATH))

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Open camera
cap = cv2.VideoCapture(0)

# Reduce camera resolution for faster processing
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

if not cap.isOpened():
    print("❌ Could not open camera")
    exit()

print("✅ Camera started!")
print("Press 'q' to quit")

while True:

    ret, frame = cap.read()

    if not ret:
        print("❌ Failed to read frame")
        break

    # YOLO inference
    results = model.predict(
        source=frame,
        conf=0.30,
        imgsz=320,       # Smaller image = faster inference
        device=device,
        verbose=False
    )

    # Draw results
    annotated_frame = results[0].plot()

    # Display
    cv2.imshow(
        "YOLO Real-Time Instance Segmentation",
        annotated_frame
    )

    # Press Q to quit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

Using device: cpu
✅ Camera started!
Press 'q' to quit


In [6]:
import cv2
from ultralytics import RTDETR

BASE_DIR = Path.cwd()
RTDETR_PATH = BASE_DIR / "rt-deter_best.pt"

try:
    model = RTDETR(RTDETR_PATH)
except Exception as e:
    print(f"Error loading model: {e}")
    exit()

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Cannot open the camera!")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model.predict(frame, conf=0.50, verbose=False)
    annotated_frame = results[0].plot()
    
    inference_time = results[0].speed.get('inference', 0)
    fps = 1000 / inference_time if inference_time > 0 else 0
    num_detections = len(results[0].boxes)
    
    cv2.putText(
        annotated_frame, 
        f"RT-DETR | FPS: {fps:.1f} | Objects: {num_detections}", 
        (10, 30), 
        cv2.FONT_HERSHEY_SIMPLEX, 
        0.7, 
        (0, 0, 255), 
        2
    )

    cv2.imshow("Live RT-DETR Detection", annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [7]:
import cv2
import numpy as np
import torch
from ultralytics import YOLO, RTDETR

# ---- Load both models once ----
YOLO_PATH = r"D:\Carrer\EM_ML_AI\Final Project 3\YOLO26n_best.pt"
RTDETR_PATH = r"D:\Carrer\EM_ML_AI\Final Project 3\rt-deter_best.pt"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

yolo_model = YOLO(YOLO_PATH)
rtdetr_model = RTDETR(RTDETR_PATH)

# ---- Open camera ----
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

if not cap.isOpened():
    print("❌ Cannot open the camera!")
    exit()

print("✅ Camera started! Press 'q' to quit.")

def run_and_annotate(model, frame, label, color):
    results = model.predict(frame, conf=0.5, imgsz=320, device=device, verbose=False)
    annotated = results[0].plot()

    inference_time = results[0].speed.get('inference', 0)
    fps = 1000 / inference_time if inference_time > 0 else 0
    num_detections = len(results[0].boxes)

    cv2.putText(
        annotated,
        f"{label} | FPS: {fps:.1f} | Obj: {num_detections}",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        color,
        2
    )
    return annotated

while True:
    ret, frame = cap.read()
    if not ret:
        print("❌ Failed to read frame")
        break

    # Run YOLO
    yolo_annotated = run_and_annotate(yolo_model, frame, "YOLO", (0, 255, 0))

    # Run RT-DETR
    rtdetr_annotated = run_and_annotate(rtdetr_model, frame, "RT-DETR", (0, 0, 255))

    # Resize both to same height (in case models output different sizes) then stack side by side
    h = 480
    yolo_resized = cv2.resize(yolo_annotated, (640, h))
    rtdetr_resized = cv2.resize(rtdetr_annotated, (640, h))

    combined = np.hstack((yolo_resized, rtdetr_resized))

    cv2.imshow("YOLO (Left)  |  RT-DETR (Right)", combined)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Using device: cpu
✅ Camera started! Press 'q' to quit.
